# 🍊 Swiggy Sales Analysis — Complete Notebook
### Old Code + New Missing Insights (Merged)
**Problem Statement:** Orders are growing but revenue is not growing proportionally. Find the root cause and recommend actions.

# 📦 Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

In [ ]:
df = pd.read_excel(r"C:\Users\HP\Downloads\swiggy_data.xlsx")
df['Order Date'] = pd.to_datetime(df['Order Date'])  # parse once here for the whole notebook

In [ ]:
df.head()

In [ ]:
df.tail(50)

# 🗂️ Meta Data

In [ ]:
print(df.shape)
print(df.shape[0])
print(df.shape[1])
print(df.info())

In [ ]:
print(df.dtypes)
df.describe()

# 📊 Building KPIs

In [ ]:
ts = df['Price (INR)'].sum()
print('Total Sales: ₹', round(ts, 2))

In [ ]:
avg_rate = df['Rating'].mean()
print('Average Rating:', round(avg_rate, 2))

In [ ]:
avg_ord_val = df['Price (INR)'].mean()
print('Average Order Value: ₹', round(avg_ord_val, 2))

In [ ]:
ratings_count = df['Rating Count'].sum()
print('Total Ratings Count:', ratings_count)

In [ ]:
total_orders = df['Restaurant Name'].count()
print('Total Orders:', total_orders)

# 📈 Chart Insights
---

## 1️⃣ Monthly Sales Trend (Original)

In [ ]:
df['Year&Month'] = df['Order Date'].dt.to_period('M').astype(str)
monthly_sale = df.groupby('Year&Month')['Price (INR)'].sum().reset_index()
print(monthly_sale)

plt.figure(figsize=(12, 5))
plt.plot(monthly_sale['Year&Month'], monthly_sale['Price (INR)'], marker='o')
plt.xticks(rotation=45)
plt.xlabel('Year & Month')
plt.ylabel('Total Sales (INR)')
plt.title('Monthly Sales Trend')
plt.tight_layout()
plt.show()

## 🔴 NEW — INSIGHT 1: AOV Over Time
**Does Average Order Value decline while orders grow? This is the smoking gun.**

In [ ]:
monthly_aov = df.groupby('Year&Month').agg(
    Total_Revenue=('Price (INR)', 'sum'),
    Total_Orders=('Order Date', 'count')
).reset_index()
monthly_aov['AOV'] = (monthly_aov['Total_Revenue'] / monthly_aov['Total_Orders']).round(2)

fig = px.line(
    monthly_aov, x='Year&Month', y='AOV',
    title='📉 Average Order Value (AOV) Over Time',
    markers=True, text='AOV'
)
fig.update_traces(textposition='top center', line_color='crimson', marker_size=8)
fig.update_layout(xaxis_title='Month', yaxis_title='AOV (INR)', title_x=0.5, height=450)
fig.show()

print(monthly_aov[['Year&Month','Total_Orders','Total_Revenue','AOV']].to_string(index=False))

# 📌 INSIGHT: If AOV is falling month over month despite order growth
# → customers are ordering cheaper dishes → revenue can't keep up with order volume.

## 🔴 NEW — INSIGHT 2: Revenue vs Orders Dual Axis
**Visual proof of the decoupling between order growth and revenue growth.**

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=monthly_aov['Year&Month'], y=monthly_aov['Total_Revenue'],
    name='Total Revenue (INR)', mode='lines+markers',
    line=dict(color='royalblue', width=2.5), yaxis='y1'
))
fig.add_trace(go.Scatter(
    x=monthly_aov['Year&Month'], y=monthly_aov['Total_Orders'],
    name='Total Orders', mode='lines+markers',
    line=dict(color='orange', width=2.5, dash='dash'), yaxis='y2'
))
fig.update_layout(
    title='📊 Revenue vs Order Count Over Time (Dual Axis)', title_x=0.5,
    xaxis_title='Month',
    yaxis=dict(title='Revenue (INR)', titlefont=dict(color='royalblue')),
    yaxis2=dict(title='Order Count', titlefont=dict(color='orange'), overlaying='y', side='right'),
    legend=dict(x=0.01, y=0.99), height=480
)
fig.show()

# 📌 INSIGHT: If orange (orders) rises faster than blue (revenue)
# → the decoupling from the problem statement is confirmed visually.

## 2️⃣ Daily Sales Trend (Original — Improved with AOV)

In [ ]:
df['Day_name'] = df['Order Date'].dt.day_name()
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

daily_summary = df.groupby('Day_name').agg(
    Total_Revenue=('Price (INR)', 'sum'),
    Total_Orders=('Day_name', 'count'),
    AOV=('Price (INR)', 'mean')
).reindex(day_order).reset_index().round(2)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=daily_summary['Day_name'], y=daily_summary['Total_Revenue'],
    name='Revenue (INR)', marker_color='steelblue', yaxis='y1'
))
fig.add_trace(go.Scatter(
    x=daily_summary['Day_name'], y=daily_summary['AOV'],
    name='AOV (INR)', mode='lines+markers+text',
    text=daily_summary['AOV'], texttemplate='₹%{text:.0f}', textposition='top center',
    line=dict(color='crimson', width=2.5), yaxis='y2'
))
fig.update_layout(
    title='📅 Daily Revenue + AOV Trend', title_x=0.5,
    xaxis_title='Day of Week',
    yaxis=dict(title='Revenue (INR)', titlefont=dict(color='steelblue')),
    yaxis2=dict(title='AOV (INR)', titlefont=dict(color='crimson'), overlaying='y', side='right'),
    height=480, legend=dict(x=0.01, y=0.99)
)
fig.show()

# 📌 INSIGHT: Days with high orders but low AOV = budget orders cluster on that day.
# Weekends should ideally have both high volume AND high AOV.

## 3️⃣ Veg vs Non-Veg Revenue Split (Original)

In [ ]:
non_veg = [
    'chicken','duck','turkey','mutton','lamb','pork','beef',
    'fish','prawn','shrimp','crab','lobster','squid','octopus',
    'sausages','bacon','salami','egg'
]
df['Food_Category'] = np.where(
    df['Dish Name'].str.lower().str.contains('|'.join(non_veg), na=False),
    'Non-Veg', 'Veg'
)
food_sales = df.groupby('Food_Category')['Price (INR)'].sum().reset_index()

pio.renderers.default = 'browser'
fig = px.pie(
    food_sales, values='Price (INR)', names='Food_Category',
    hole=0.5, title='Revenue Distribution by Food Category'
)
fig.update_traces(
    textinfo='percent+label',
    pull=[0.05] * len(food_sales['Food_Category'].unique())
)
fig.update_layout(height=500, margin=dict(t=60, b=40, l=40, r=40))
fig.show()

## 4️⃣ Revenue by State (Original)

In [ ]:
state_wise_sale = df.groupby('State')['Price (INR)'].sum().reset_index()
print(state_wise_sale)

fig = px.bar(
    state_wise_sale.sort_values('Price (INR)', ascending=False),
    y='State', x='Price (INR)', orientation='h',
    title='Revenue by State (INR)'
)
fig.update_layout(height=600, yaxis=dict(autorange='reversed'))
fig.show()

## 🟡 NEW — INSIGHT 7: City-wise AOV — Which Cities Pull Average Down?

In [ ]:
city_summary = df.groupby('City').agg(
    Total_Revenue=('Price (INR)', 'sum'),
    Total_Orders=('City', 'count'),
    AOV=('Price (INR)', 'mean')
).reset_index().round(2)
city_summary = city_summary.sort_values('AOV', ascending=False)

overall_aov = df['Price (INR)'].mean()
city_summary['Color'] = city_summary['AOV'].apply(
    lambda x: 'Above Average' if x >= overall_aov else 'Below Average'
)

fig = px.bar(
    city_summary, x='AOV', y='City', orientation='h',
    color='Color',
    color_discrete_map={'Above Average': 'seagreen', 'Below Average': 'crimson'},
    text='AOV',
    title=f'🏙️ City-wise AOV vs National Average (₹{overall_aov:.0f})'
)
fig.add_vline(x=overall_aov, line_dash='dash', line_color='black',
              annotation_text=f'Avg ₹{overall_aov:.0f}', annotation_position='top right')
fig.update_traces(texttemplate='₹%{text:.0f}', textposition='outside')
fig.update_layout(height=650, title_x=0.5)
fig.show()

print('\nBottom 5 Cities by AOV:')
print(city_summary.tail(5)[['City','Total_Orders','Total_Revenue','AOV']].to_string(index=False))

# 📌 INSIGHT: Red cities prefer cheaper dishes → recommend premium restaurant onboarding there.

## 5️⃣ Quarterly Sales Summary (Original — Enhanced with AOV & Growth %)

In [ ]:
df['Quarterly'] = df['Order Date'].dt.to_period('Q').astype(str)

quarterly_full = df.groupby('Quarterly', as_index=False).agg(
    Total_Revenue=('Price (INR)', 'sum'),
    Total_Orders=('Order Date', 'count'),
    Avg_Rating=('Rating', 'mean')
).sort_values('Quarterly')

quarterly_full['AOV'] = (quarterly_full['Total_Revenue'] / quarterly_full['Total_Orders']).round(2)
quarterly_full['Revenue_Growth_%'] = quarterly_full['Total_Revenue'].pct_change().mul(100).round(2)
quarterly_full['Order_Growth_%'] = quarterly_full['Total_Orders'].pct_change().mul(100).round(2)

print('📊 FULL QUARTERLY SUMMARY:')
print(quarterly_full.to_string(index=False))

fig = go.Figure()
fig.add_trace(go.Bar(
    x=quarterly_full['Quarterly'], y=quarterly_full['Total_Revenue'],
    name='Revenue', marker_color='steelblue', yaxis='y1'
))
fig.add_trace(go.Scatter(
    x=quarterly_full['Quarterly'], y=quarterly_full['AOV'],
    name='AOV', mode='lines+markers+text',
    text=quarterly_full['AOV'], texttemplate='₹%{text}', textposition='top center',
    line=dict(color='crimson', width=2.5), yaxis='y2'
))
fig.update_layout(
    title='📆 Quarterly Revenue + AOV', title_x=0.5,
    yaxis=dict(title='Revenue (INR)', titlefont=dict(color='steelblue')),
    yaxis2=dict(title='AOV (INR)', titlefont=dict(color='crimson'), overlaying='y', side='right'),
    height=450
)
fig.show()

# 📌 INSIGHT: If Order_Growth% > Revenue_Growth% every quarter → structural AOV problem.

## 6️⃣ Top 5 Cities by Revenue (Original)

In [ ]:
df['Price (INR)'] = pd.to_numeric(df['Price (INR)'], errors='coerce')
df = df.dropna(subset=['Price (INR)', 'City'])

city_sale = (
    df.groupby('City', as_index=False)['Price (INR)']
    .sum().sort_values('Price (INR)', ascending=False).head(5)
    .sort_values('Price (INR)', ascending=True)
)

fig = px.bar(
    city_sale, x='Price (INR)', y='City', orientation='h',
    title='Top 5 Cities by Revenue (INR)', text='Price (INR)',
    color='Price (INR)', color_continuous_scale='Reds'
)
fig.update_traces(texttemplate='%{text:.2s}', textposition='outside')
fig.update_layout(
    height=500, margin=dict(t=60, b=40, l=40, r=40),
    xaxis_title='Revenue (INR)', yaxis_title='City', title_x=0.5
)
fig.show()

## 7️⃣ Weekly Sales Trend (Original)

In [ ]:
df['week'] = df['Order Date'].dt.to_period('W').astype(str)
weekly_sales = df.groupby('week', as_index=False)['Price (INR)'].sum()

fig = px.line(
    weekly_sales, x='week', y='Price (INR)',
    title='Weekly Sales Trend (INR)', markers=True
)
fig.update_layout(xaxis_title='Week', yaxis_title='Revenue (INR)', title_x=0.5)
fig.show()

---
# 🔴 NEW INSIGHTS — Root Cause Analysis
---

## 🔴 NEW — INSIGHT 3: Category-Level Revenue, Orders & AOV
**Which categories drive revenue? Which have high orders but low AOV (cheap drag)?**

In [ ]:
category_summary = df.groupby('Category').agg(
    Total_Revenue=('Price (INR)', 'sum'),
    Total_Orders=('Category', 'count'),
    AOV=('Price (INR)', 'mean')
).reset_index()
category_summary['AOV'] = category_summary['AOV'].round(2)
category_summary = category_summary.sort_values('Total_Revenue', ascending=False)

top15 = category_summary.head(15)
fig1 = px.bar(
    top15.sort_values('Total_Revenue', ascending=True),
    x='Total_Revenue', y='Category', orientation='h',
    title='🏆 Top 15 Categories by Revenue', text='Total_Revenue',
    color='AOV', color_continuous_scale='RdYlGn'
)
fig1.update_traces(texttemplate='₹%{text:.2s}', textposition='outside')
fig1.update_layout(height=550, title_x=0.5)
fig1.show()

overall_aov = df['Price (INR)'].mean()
cheap_drag = category_summary[
    (category_summary['Total_Orders'] > category_summary['Total_Orders'].quantile(0.6)) &
    (category_summary['AOV'] < overall_aov)
].sort_values('Total_Orders', ascending=False).head(10)

print(f'Overall AOV: ₹{overall_aov:.2f}')
print('\n⚠️ HIGH ORDER but LOW AOV Categories (Revenue Drag):')
print(cheap_drag[['Category','Total_Orders','Total_Revenue','AOV']].to_string(index=False))

# 📌 INSIGHT: These categories get lots of orders but pull AOV down.
# Action: Bundle low-AOV items or promote higher-value dishes within these categories.

## 🔴 NEW — INSIGHT 4: Price Tier Distribution
**Are budget orders (₹0–100) dominating volume and suppressing revenue?**

In [ ]:
bins = [0, 100, 200, 300, 500, 8001]
labels = ['₹0-100 (Budget)', '₹100-200 (Economy)', '₹200-300 (Mid)', '₹300-500 (Premium)', '₹500+ (Luxury)']
df['Price_Tier'] = pd.cut(df['Price (INR)'], bins=bins, labels=labels, include_lowest=True)

tier_summary = df.groupby('Price_Tier', observed=True).agg(
    Total_Orders=('Price (INR)', 'count'),
    Total_Revenue=('Price (INR)', 'sum')
).reset_index()
tier_summary['Revenue_Share_%'] = (tier_summary['Total_Revenue'] / tier_summary['Total_Revenue'].sum() * 100).round(1)
tier_summary['Order_Share_%'] = (tier_summary['Total_Orders'] / tier_summary['Total_Orders'].sum() * 100).round(1)

fig = go.Figure()
fig.add_trace(go.Bar(
    name='Order Share %', x=tier_summary['Price_Tier'], y=tier_summary['Order_Share_%'],
    marker_color='steelblue', text=tier_summary['Order_Share_%'],
    texttemplate='%{text}%', textposition='outside'
))
fig.add_trace(go.Bar(
    name='Revenue Share %', x=tier_summary['Price_Tier'], y=tier_summary['Revenue_Share_%'],
    marker_color='coral', text=tier_summary['Revenue_Share_%'],
    texttemplate='%{text}%', textposition='outside'
))
fig.update_layout(
    barmode='group',
    title='💰 Order Share vs Revenue Share by Price Tier',
    title_x=0.5, xaxis_title='Price Tier', yaxis_title='Share (%)', height=480
)
fig.show()

print(tier_summary.to_string(index=False))

# 📌 INSIGHT: If Budget tier Order% >> Revenue% → cheap orders dominate volume.
# The gap between bars per tier shows where the revenue leak is hiding.

## 🟡 NEW — INSIGHT 5: Rating Impact on Revenue
**Do higher-rated restaurants earn more per order?**

In [ ]:
restaurant_summary = df.groupby('Restaurant Name').agg(
    Avg_Rating=('Rating', 'mean'),
    AOV=('Price (INR)', 'mean'),
    Total_Revenue=('Price (INR)', 'sum'),
    Total_Orders=('Restaurant Name', 'count'),
    Avg_Rating_Count=('Rating Count', 'mean')
).reset_index().round(2)

restaurant_filtered = restaurant_summary[restaurant_summary['Avg_Rating_Count'] >= 10]

fig = px.scatter(
    restaurant_filtered,
    x='Avg_Rating', y='AOV',
    size='Total_Revenue', color='Total_Orders',
    hover_name='Restaurant Name',
    color_continuous_scale='Viridis',
    title='⭐ Restaurant Rating vs AOV (bubble size = Total Revenue)',
    labels={'Avg_Rating': 'Average Rating', 'AOV': 'Avg Order Value (INR)'}
)
fig.update_layout(title_x=0.5, height=520)
fig.show()

corr = restaurant_filtered[['Avg_Rating', 'AOV']].corr().iloc[0, 1]
print(f'Correlation between Rating and AOV: {corr:.3f}')
print('(+1 = higher rated charge more | 0 = no relation)')

# 📌 INSIGHT: Positive correlation → promote high-rated restaurants for better revenue.
# Near 0 → ratings alone don't drive premium pricing on Swiggy.

## 🟡 NEW — INSIGHT 6: Top 10 Restaurants — Revenue vs Orders
**Do high-order restaurants also bring high revenue?**

In [ ]:
top10_revenue = restaurant_summary.nlargest(10, 'Total_Revenue')[['Restaurant Name','Total_Revenue','Total_Orders','AOV']]
top10_orders = restaurant_summary.nlargest(10, 'Total_Orders')[['Restaurant Name','Total_Revenue','Total_Orders','AOV']]

print('🏆 TOP 10 BY REVENUE:')
print(top10_revenue.to_string(index=False))
print('\n📦 TOP 10 BY ORDER COUNT:')
print(top10_orders.to_string(index=False))

fig = px.bar(
    top10_revenue.sort_values('Total_Revenue'),
    x='Total_Revenue', y='Restaurant Name', orientation='h',
    color='AOV', text='AOV', color_continuous_scale='Blues',
    title='🏆 Top 10 Restaurants by Revenue (color = AOV)'
)
fig.update_traces(texttemplate='AOV: ₹%{text:.0f}', textposition='inside')
fig.update_layout(height=500, title_x=0.5)
fig.show()

# 📌 INSIGHT: Restaurants high in orders but low in revenue = low-AOV traffic magnets.
# Action: Swiggy should surface high-AOV restaurants more in search/homepage.

## 🟢 NEW — INSIGHT 9: Unverified Ratings Check (Data Quality)
**How much of our data has Rating Count = 0? Should we exclude it from insights?**

In [ ]:
total = len(df)
zero_rc = (df['Rating Count'] == 0).sum()
pct = round(zero_rc / total * 100, 2)

print(f'Total records: {total:,}')
print(f'Rating Count = 0 (unverified): {zero_rc:,} ({pct}%)')
print(f'Verified ratings: {total - zero_rc:,} ({100-pct}%)')

rev_unverified = df[df['Rating Count'] == 0]['Price (INR)'].sum()
rev_verified = df[df['Rating Count'] > 0]['Price (INR)'].sum()

fig = px.pie(
    values=[rev_unverified, rev_verified],
    names=['Unverified (Count=0)', 'Verified (Count>0)'],
    hole=0.4, title='🔍 Revenue: Verified vs Unverified Dishes',
    color_discrete_sequence=['#f4a261', '#2a9d8f']
)
fig.update_traces(textinfo='percent+label')
fig.update_layout(title_x=0.5, height=420)
fig.show()

# 📌 INSIGHT: If unverified dishes = significant revenue → we have a data gap.
# Rating-based analysis should filter Rating Count = 0 or flag as 'insufficient data'.